In [ ]:
import os 
import ast
import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
from wquantiles import quantile
from statsmodels.stats.weightstats import DescrStatsW
from openfisca_france_indirect_taxation import FranceIndirectTaxationTaxBenefitSystem
from openfisca_france_indirect_taxation.surveys import SurveyScenario
from openfisca_france_indirect_taxation.utils import assets_directory, get_input_data_frame
from openfisca_france_indirect_taxation.projects.TVA.Utils import weighted_quantiles 
from openfisca_france_indirect_taxation.build_survey_data.utils import collapsesum
from openfisca_france_indirect_taxation.Calage_revenus_bdf import compute_erfs_decile, calage_bdf_niveau_vie
from openfisca_france_indirect_taxation.Calage_consommation_bdf import get_inflators_by_year
from openfisca_france_indirect_taxation.examples.utils_example import df_weighted_average_grouped, wavg, collapse
from openfisca_france_indirect_taxation.projects.PhD_project_Herve.Fuel_tax_reform import reform_ticpe_2019_in_2018


In [ ]:
year = 2018
data_year = 2017
tax_benefit_system = FranceIndirectTaxationTaxBenefitSystem()
inflators_by_year = get_inflators_by_year(rebuild = False, year_range = range(2017, 2025), data_year = data_year)
inflation_kwargs = dict(inflator_by_variable = inflators_by_year[year])
inflation_kwargs.get('inflator_by_variable').update({'depenses_carburants_entree' : inflation_kwargs.get('inflator_by_variable').pop('depenses_carburants'),
                                                     'depense_gazole_total_ttc_entree' : inflation_kwargs.get('inflator_by_variable').pop('depenses_diesel'),
                                                     'depense_essence_total_ttc_entree' : inflation_kwargs.get('inflator_by_variable').pop('depenses_essence'),})

In [ ]:
input_bdf = get_input_data_frame(2017, use_emp = True)
input_bdf.rename({'depenses_carburants' : 'depenses_carburants_entree',
                  'depenses_diesel' : 'depense_gazole_total_ttc_entree',
                  'depenses_essence' : 'depense_essence_total_ttc_entree'}, axis = 1, inplace= True)
input_bdf = input_bdf.loc[input_bdf['rev_disponible'] > 0]

erfs_path = 'C:/Users/veve1/OneDrive/Documents/ENSAE 3A/Memoire MiE/Data/erfs_fpr/{}/csv'.format(year) 
erfs_menage_by_decile = compute_erfs_decile(year, 'men', erfs_path)
erfs_menage_by_decile

input_bdf , df_calage = calage_bdf_niveau_vie(input_bdf, erfs_menage_by_decile, 'men')

labels = np.arange(1, 5)
input_bdf['depenses_carburants_entree'] = input_bdf['depenses_carburants_entree'].astype(float)
input_bdf['quartile_depenses_carburants'] = weighted_quantiles(input_bdf['depenses_carburants_entree'], labels, input_bdf['pondmen'], return_quantiles = False)

In [ ]:
survey_scenario = SurveyScenario.create(
        input_data_frame = input_bdf,           # Les niveaux de vie sont calés sur ceux de l'ERFS 2018 (pour des déciles de ménages)
        inflation_kwargs =  inflation_kwargs,
        baseline_tax_benefit_system = tax_benefit_system,
        reform = reform_ticpe_2019_in_2018,
        period = year,
        )

In [ ]:
simulated_variables = [
    'identifiant_menage',
    'taxes_carburant_total',
    'emissions_CO2_carburants',
    'rev_disponible',
    'ocde10',
    'pondmen',
    'niveau_de_vie',
    'niveau_vie_decile',
    'redistribution_reform',
    'quartile_depenses_carburants',
    'elas_exp_1']

In [ ]:
baseline_menage = survey_scenario.create_data_frame_by_entity(simulated_variables, use_baseline = True, period = year)['menage']
reform_menage = survey_scenario.create_data_frame_by_entity(simulated_variables, use_baseline = False, period = year)['menage']

# On calcule la différence entre la situation de référence et la situation avec réforme pour les variables d'intérêt
difference_menage = baseline_menage - reform_menage
variables = ['rev_disponible','niveau_de_vie','niveau_vie_decile','ocde10','pondmen', 'quartile_depenses_carburants','identifiant_menage']
difference_menage.loc[:, variables ] = baseline_menage.loc[:, variables]
difference_menage.loc[:, 'ref_elasticity'] = "No response"

difference_menage_no_response = difference_menage.loc[: , ['identifiant_menage', 
                                                           'rev_disponible',
                                                           'niveau_de_vie',
                                                           'niveau_vie_decile',
                                                           'ocde10',
                                                           'pondmen', 
                                                           'quartile_depenses_carburants',
                                                           'taxes_carburant_total']]

In [ ]:
difference_menage_no_response

In [ ]:
# Create a table with elasticities from Bonnet et al. (2025)
df_elas = pd.DataFrame(data = { 'quartile_depenses_carburants' : ['Elasticity'],
                                   1.0 : [-0.82] , 
                                   2.0 : [-0.35] , 
                                   3.0 : [- 0.35] , 
                                   4.0 : [- 0.26] })
df_elas = df_elas.T
df_elas.reset_index(inplace= True)
df_elas.rename(columns = df_elas.iloc[0], inplace = True)
df_elas.drop(index = 0, axis = 0, inplace = True)
df_elas['ref_elasticity'] = 'Bonnet (2025) quartiles'
df_elas['quartile_depenses_carburants'] = df_elas['quartile_depenses_carburants'].astype(float)

input_bdf = input_bdf.merge(right = df_elas, how = 'left', on = 'quartile_depenses_carburants')
input_bdf['elas_exp_1'] = input_bdf['Elasticity']
input_bdf.drop('Elasticity', axis = 1, inplace = True)

In [ ]:
survey_scenario = SurveyScenario.create(
    input_data_frame = input_bdf,           # Les niveaux de vie sont calés sur ceux de l'ERFS 2018 (pour des déciles de ménages !)
    inflation_kwargs = inflation_kwargs,
    baseline_tax_benefit_system = tax_benefit_system,
    reform = reform_ticpe_2019_in_2018,
    period = year,
    )

baseline_menage = survey_scenario.create_data_frame_by_entity(simulated_variables, use_baseline = True, period = year)['menage']
reform_menage = survey_scenario.create_data_frame_by_entity(simulated_variables, use_baseline = False, period = year)['menage']

# On calcule la différence entre la situation de référence et la situation avec réforme pour les variables d'intérêt
difference_menage = baseline_menage - reform_menage
variables = ['identifiant_menage']
difference_menage.loc[:, variables ] = baseline_menage.loc[:, variables]

# On calcule la redistribution du produit de la réforme
delta_taxes_caburant_total = survey_scenario.compute_aggregate('taxes_carburant_total', aggfunc='sum', difference= True, period = year)
pondmen_total = survey_scenario.compute_aggregate('pondmen', aggfunc='sum', use_baseline= True, weighted= False, period = year)
difference_menage.loc[:, 'redistribution_reform'] = delta_taxes_caburant_total / pondmen_total

# On calcule la reduction d'emissions de CO2 totale (en tonnes)
delta_CO2_emissions_total = survey_scenario.compute_aggregate('emissions_CO2_carburants', aggfunc='sum', difference= True, period = year) 
difference_menage.loc[:, ['reduction_emissions_CO2']] = delta_CO2_emissions_total / (pondmen_total *  1E3)

difference_menage_response = difference_menage.loc[: , ['identifiant_menage','redistribution_reform','reduction_emissions_CO2']]
indirect_utility = difference_menage_no_response.merge(right = difference_menage_response, how = 'left', on = 'identifiant_menage')

# On calcule les gagnants / perdants
indirect_utility['social_cost_carbon'] = 55
indirect_utility['Net_transfer_reform'] = indirect_utility[['taxes_carburant_total','redistribution_reform']].sum(axis = 1)
indirect_utility['total_change_utility'] = indirect_utility['Net_transfer_reform'] - (indirect_utility['reduction_emissions_CO2'] * indirect_utility['social_cost_carbon'])
indirect_utility['is_winner'] = (indirect_utility['total_change_utility'] >= 0)

## Vertical effects

In [ ]:
# On crée une dataframe avec les moyennes pondérées par déciles de ménage (difference = True)
varlist = ['rev_disponible','niveau_de_vie','ocde10','taxes_carburant_total','redistribution_reform','Net_transfer_reform','reduction_emissions_CO2','total_change_utility','is_winner']
indirect_utility_by_decile = df_weighted_average_grouped(indirect_utility,
                                                         'niveau_vie_decile',
                                                         varlist,
                                                         'pondmen')
indirect_utility_by_decile['pondmen'] = indirect_utility.groupby('niveau_vie_decile')['pondmen'].sum()
indirect_utility_by_decile['social_cost_carbon'] = 55

# On calcul les totaux (additional taxes over disp income, net transfer, net transfer over disp income, change in utility, change in utility over disp income)
indirect_utility_by_decile['taxes_carburant_sur_revenu'] = indirect_utility_by_decile['taxes_carburant_total'] / indirect_utility_by_decile['rev_disponible'] * 100
indirect_utility_by_decile['Net_transfer_sur_revenu'] = indirect_utility_by_decile['Net_transfer_reform'] / indirect_utility_by_decile['rev_disponible'] * 100
indirect_utility_by_decile['total_change_utility_sur_revenu'] = indirect_utility_by_decile['total_change_utility'] / indirect_utility_by_decile['rev_disponible'] * 100
indirect_utility_by_decile['is_loser'] = 1 - indirect_utility_by_decile['is_winner']

In [ ]:
# On calcule les valeurs moyennes
Avg_taxes_carburant_total = wavg(indirect_utility_by_decile, 'taxes_carburant_total', 'pondmen')
Avg_taxes_carburant_revenu =  Avg_taxes_carburant_total / wavg(indirect_utility_by_decile, 'rev_disponible', 'pondmen') * 100
    
Avg_net_transfer_reform = wavg(indirect_utility_by_decile, 'Net_transfer_reform', 'pondmen')
Avg_net_transfer_revenu = Avg_net_transfer_reform / wavg(indirect_utility_by_decile, 'rev_disponible', 'pondmen') * 100

Avg_total_change_utility = wavg(indirect_utility_by_decile, 'total_change_utility', 'pondmen')
Avg_total_change_utility_sur_revenu = Avg_total_change_utility / wavg(indirect_utility_by_decile, 'rev_disponible', 'pondmen') * 100

In [ ]:
output_path = "C:/Users/veve1/OneDrive/Documents/ENSAE PhD/Carbon tax/Output/Figures/Political_feasibility"

In [ ]:
indirect_utility_by_decile

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = indirect_utility_by_decile, x='niveau_vie_decile', y='taxes_carburant_total', color = sns.color_palette("Paired")[0])
plt.axhline(y = Avg_taxes_carburant_total, ls = '--', color = 'orange', linewidth = 2.5)
plt.text(
    x = 2.2, 
    y = -68 ,
    s = f'Mean = {Avg_taxes_carburant_total:.1f} €',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 18,
    color = 'orange'
)
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Net effect per household (in €)', size = 18)
plt.xticks(fontsize = 18)
plt.yticks(  fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
# plt.savefig(os.path.join(output_path,'Additional_taxes.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = indirect_utility_by_decile, x='niveau_vie_decile', y='taxes_carburant_sur_revenu', color = sns.color_palette("Paired")[0])
plt.axhline(y = Avg_taxes_carburant_revenu, ls = '--', color = 'orange', linewidth = 2.5)
plt.text(
    x = 9.1, 
    y = - 0.19,
    s = f'Mean = {Avg_taxes_carburant_revenu:.2f} %',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 18,
    color = 'orange'
)
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Additional taxes (% of disp income)', size = 16)
plt.xticks(fontsize = 18)
plt.yticks(  fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
# plt.savefig(os.path.join(output_path,'Additional_taxes_sur_revenu.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = indirect_utility_by_decile, x='niveau_vie_decile', y='Net_transfer_reform', color = sns.color_palette("Paired")[0])
plt.axhline(y = Avg_net_transfer_reform, ls = '--', color = 'orange', linewidth = 2.5)
plt.text(
    x = 2.2, 
    y = -15.2 ,
    s = f'Mean = {Avg_net_transfer_reform:.1f} €',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 18,
    color = 'orange'
)
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Net effect per household (in €)', size = 18)
plt.xticks(fontsize = 18)
plt.yticks(np.arange(-30,25,5),  fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
# plt.savefig(os.path.join(output_path,'Net_transfer_reform.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = indirect_utility_by_decile, x='niveau_vie_decile', y='Net_transfer_sur_revenu', color = sns.color_palette("Paired")[0])
plt.axhline(y = Avg_net_transfer_revenu, ls = '--', color = 'orange', linewidth = 2.5)
plt.text(
    x = 2.2, 
    y = -0.052 ,
    s = f'Mean = {Avg_net_transfer_revenu:.2f} %',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 18,
    color = 'orange'
)
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Net effect (% of household disp income)', size = 16)
plt.xticks(fontsize = 18)
plt.yticks(np.arange(-.1,0.20,0.05),  fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
# plt.savefig(os.path.join(output_path,'Net_transfer_revenu.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = indirect_utility_by_decile, x='niveau_vie_decile', y='total_change_utility', color = sns.color_palette("Paired")[0])
plt.axhline(y = Avg_total_change_utility, ls = '--', color = 'orange', linewidth = 2.5)
plt.text(
    x = 2, 
    y = -10.3,
    s = f'Mean = {Avg_total_change_utility:.1f} €',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 18,
    color = 'orange'
)
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Total change in utility (in €)', size = 18)
plt.xticks(fontsize = 18)
plt.yticks(np.arange(-25,30,5),  fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(output_path,'Total_change_utility.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = indirect_utility_by_decile, x='niveau_vie_decile', y='total_change_utility_sur_revenu', color = sns.color_palette("Paired")[0], width = 0.9)
plt.axhline(y = Avg_total_change_utility_sur_revenu, ls = '--', color = 'orange', linewidth = 2.5)
plt.text(
    x = 2.2, 
    y = - 0.04 ,
    s = f'Mean = {Avg_total_change_utility_sur_revenu:.2f} %',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 18,
    color = 'orange'
)
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Net effect (% of household disp income)', size = 16)
plt.xticks(fontsize = 18)
plt.yticks(np.arange(-.1,0.25,0.05),  fontsize = 16)
plt.grid(True, linestyle='--', alpha = 0.7)
plt.savefig(os.path.join(output_path,'Total_change_utility_sur_revenu.pdf'), bbox_inches = 'tight')

In [ ]:
share_winners = wavg(indirect_utility_by_decile, 'is_winner', 'pondmen') * 100

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = indirect_utility_by_decile, x='niveau_vie_decile', y=1, color = sns.color_palette("Paired")[5], alpha = 0.8)
ax = sns.barplot(data = indirect_utility_by_decile, x='niveau_vie_decile', y='is_winner', color = sns.color_palette("Paired")[3])
plt.axhline(y = share_winners / 100, ls = '--', color = 'black', linewidth = 2.5)
plt.text(
    x = 4.8, 
    y = 0.47,
    s = f'Total Share of winners = {share_winners:.1f} %',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 18,
    color = 'black'
)
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Share of winners/losers', size = 18)
plt.xticks(fontsize = 18)
plt.yticks(  fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(output_path,'Share_winners_losers.pdf'), bbox_inches = 'tight')

## Horizontal effects

In [ ]:
indirect_utility_by_groups_by_decile = indirect_utility.groupby([
    'quartile_depenses_carburants', 
    'niveau_vie_decile'])[
        ['pondmen',
         'rev_disponible',
         'taxes_carburant_total',
         'reduction_emissions_CO2',
         'redistribution_reform',
         'Net_transfer_reform',
         'total_change_utility']].apply(
    lambda g: pd.Series({
        'pondmen': g['pondmen'].sum(),
        'rev_disponible': np.average(g['rev_disponible'], weights=g['pondmen']),
        'taxes_carburant_total': np.average(g['taxes_carburant_total'], weights=g['pondmen']),
        'reduction_emissions_CO2': np.average(g['reduction_emissions_CO2'], weights=g['pondmen']),
        'redistribution_reform': np.average(g['redistribution_reform'], weights=g['pondmen']),
        'Net_transfer_reform': np.average(g['Net_transfer_reform'], weights=g['pondmen']),
        'total_change_utility': np.average(g['total_change_utility'], weights=g['pondmen']),
    })   
)
indirect_utility_by_groups_by_decile.reset_index(inplace = True)
indirect_utility_by_groups_by_decile['pond_decile'] = indirect_utility_by_groups_by_decile.groupby('niveau_vie_decile')['pondmen'].transform('sum')
indirect_utility_by_groups_by_decile['share'] = indirect_utility_by_groups_by_decile['pondmen'] / indirect_utility_by_groups_by_decile['pond_decile']

indirect_utility_by_groups_by_decile['social_cost_carbon'] = 55
# Compute variables
indirect_utility_by_groups_by_decile['taxes_carburant_sur_revenu'] = indirect_utility_by_groups_by_decile['taxes_carburant_total'] / indirect_utility_by_groups_by_decile['rev_disponible'] * 100
indirect_utility_by_groups_by_decile['Net_transfer_sur_revenu'] = indirect_utility_by_groups_by_decile['Net_transfer_reform'] / indirect_utility_by_groups_by_decile['rev_disponible'] * 100
indirect_utility_by_groups_by_decile['total_change_utility_sur_revenu'] = indirect_utility_by_groups_by_decile['total_change_utility'] / indirect_utility_by_groups_by_decile['rev_disponible'] * 100

In [ ]:
palette = sns.color_palette("coolwarm",6)[0:2] + sns.color_palette("coolwarm",6)[4:6]

In [ ]:
indirect_utility_by_groups_by_decile.loc[indirect_utility_by_groups_by_decile['quartile_depenses_carburants'] == 4]

In [ ]:
grouped_share = indirect_utility_by_groups_by_decile.reset_index()
# pivot to get shares per quartile per decile and fill missing with 0
pivot_share = grouped_share.pivot(index='niveau_vie_decile', columns='quartile_depenses_carburants', values='share').fillna(0)

plt.figure(figsize=(10, 6))
colors = palette
bottom = np.zeros(len(pivot_share))
for i, col in enumerate(pivot_share.columns):
    plt.bar(pivot_share.index, pivot_share[col], bottom=bottom, color=colors[i], width=0.8 , label=f'Q{col}')
    bottom += pivot_share[col].values
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Share of households', fontsize=18)
plt.xticks(np.arange(1,11,1),fontsize = 18)
plt.yticks(fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(loc = 'upper right', ncols = 4, title='Fuel expenses quartile', fontsize=16, title_fontsize=16)
plt.margins(x = 0.01)
plt.savefig(os.path.join(output_path,'Share_households_quartiles_fuel.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
plt.axhline(y = 0, ls = '-', color = 'black', linewidth = 1)
sns.scatterplot(x = 'niveau_vie_decile', 
                y = 'Net_transfer_reform', 
                hue = 'quartile_depenses_carburants',
                palette = palette,
                s = 160,
                data = indirect_utility_by_groups_by_decile,
                )
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Net effect (in €)', size = 18)
plt.xticks(np.arange(1,11,1), fontsize = 18)
plt.yticks(np.arange(-150,175, 50),  fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(loc = 'upper right', ncols = 4, title='Fuel expenses quartile', fontsize=16, title_fontsize=16)
# plt.savefig(os.path.join(output_path,'Net_transfer_reform_by_quartile.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
plt.axhline(y = 0, ls = '-', color = 'black', linewidth = 1)
sns.scatterplot(x = 'niveau_vie_decile', 
                y = 'Net_transfer_sur_revenu', 
                hue = 'quartile_depenses_carburants',
                palette = palette,
                s = 160,
                data = indirect_utility_by_groups_by_decile,
                )
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Net effect (% of household disp income)', size = 16)
plt.xticks(np.arange(1,11,1), fontsize = 18)
plt.yticks(np.arange(-.8,0.8, 0.2),  fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(loc = 'upper right', ncols = 4, title='Fuel expenses quartile', fontsize=16, title_fontsize=16)
plt.savefig(os.path.join(output_path,'Net_transfer_sur_revenu_by_quartile.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
plt.axhline(y = 0, ls = '--', color = 'black', linewidth = 1)
sns.scatterplot(x = 'niveau_vie_decile', 
                y = 'total_change_utility', 
                hue = 'quartile_depenses_carburants',
                palette = palette,
                s = 150,
                data = indirect_utility_by_groups_by_decile,
                )
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Net effect (% of household disp income)', size = 16)
plt.xticks(np.arange(1,11,1), fontsize = 18)
plt.yticks(np.arange(-150,175, 50),  fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(loc = 'upper right', ncols = 4, title='Fuel expenses quartile', fontsize=16, title_fontsize=16)
plt.savefig(os.path.join(output_path,'Total_change_utility_by_quartile.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
plt.axhline(y = 0, ls = '--', color = 'black', linewidth = 1)
sns.scatterplot(x = 'niveau_vie_decile', 
                y = 'total_change_utility_sur_revenu', 
                hue = 'quartile_depenses_carburants',
                palette = palette,
                s = 150,
                data = indirect_utility_by_groups_by_decile,
                )
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Net effect (% of household disp income)', size = 16)
plt.xticks(np.arange(1,11,1), fontsize = 18)
plt.yticks(np.arange(-.8,0.7, 0.2),  fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(loc = 'upper right', ncols = 4, title='Fuel expenses quartile', fontsize=16, title_fontsize=16)
plt.savefig(os.path.join(output_path,'Total_change_utility_sur_revenu_by_quartile.pdf'), bbox_inches = 'tight')

## Social welfare

In [ ]:
# Vertical heterogeneity
indirect_utility['Rawlsian(p10)'] = (indirect_utility['niveau_vie_decile'] == 1).astype(float)* indirect_utility['total_change_utility']
indirect_utility['Decreasing (a = 0.8)'] = indirect_utility['rev_disponible'].apply(lambda x : x**(-0.8)) * indirect_utility['total_change_utility']
indirect_utility['Decreasing (a = 1)'] = indirect_utility['rev_disponible'].apply(lambda x :x**(-1)) * indirect_utility['total_change_utility']
indirect_utility['Decreasing (a = 2)'] = indirect_utility['rev_disponible'].apply(lambda x :x**(-2)) * indirect_utility['total_change_utility']
# Horizontal heterogeneity
indirect_utility['Car dependant (Q4)'] = (indirect_utility['quartile_depenses_carburants'] == 4).astype(float) * indirect_utility['total_change_utility']

# Both
indirect_utility['Rawlsian & car dependant (Q4)'] = (indirect_utility['niveau_vie_decile'] == 1).astype(float) * indirect_utility['Car dependant (Q4)']
indirect_utility['coef_quartile_carburants'] = (indirect_utility['quartile_depenses_carburants'] == 4).astype(float) * 0.5 + (
    indirect_utility['quartile_depenses_carburants'] == 3).astype(float) * 1 + (
        indirect_utility['quartile_depenses_carburants'] == 2).astype(float) * 1.5 + (
            indirect_utility['quartile_depenses_carburants'] == 1).astype(float) * 2
indirect_utility['Vertical & Horizontal'] = indirect_utility['rev_disponible'].pow(-indirect_utility['coef_quartile_carburants']) * indirect_utility['total_change_utility']

In [ ]:
indirect_utility['Group'] = 'All'

In [ ]:
welfare_varlist = ['total_change_utility','Rawlsian(p10)','Decreasing (a = 0.8)','Decreasing (a = 1)','Decreasing (a = 2)','Car dependant (Q4)','Rawlsian & car dependant (Q4)','Vertical & Horizontal']
welfare = df_weighted_average_grouped(indirect_utility, 'Group', welfare_varlist, 'pondmen').reset_index().drop(columns = 'Group', axis = 0)
welfare.apply(lambda x : x >0)

In [ ]:
welfare_latex_table = pd.DataFrame({
    'Social welfare function': [
        'Rawlsian(p10)',
        'Decreasing (a = 0.8)',
        'Decreasing (a = 1)',
        'Decreasing (a = 2)',
        'Car dependant (Q4)',
        'Rawlsian & car dependant (Q4)',
        'Vertical & Horizontal',
    ],
    'Welfare weights': [
        '$\forall z \forall \theta, g(z, \theta) = 1$ if $z$ \in p10, 0 otherwise',
        '$\forall z \forall \theta, g(z, \theta) = z^{-0.8}$',
        '$\forall z \forall \theta, g(z, \theta) = z^{-1}$',
        '$\forall z \forall \theta, g(z, \theta) = z^{-2}$',
        '$\forall z \forall \theta, g(z, \theta) = 1$ if $z$ \in Q4, 0 otherwise',
        '$\forall z \forall \theta, g(z, \theta) = 1$ if $z$ \in p1 and $z$ \in Q4, 0 otherwise',
        '$\forall z \forall \theta, g(z, \theta) = z^{-coef_quartile_carburants} with coef = 0.5 (Q4), 1 (Q3), 1.5 (Q2), 2 (Q1)',
    ],
    'Result': [
        'Positive' if welfare.iloc[0]['Rawlsian(p10)'] >= 0 else 'Negative',
        'Positive' if welfare.iloc[0]['Decreasing (a = 0.8)'] >= 0 else 'Negative',
        'Positive' if welfare.iloc[0]['Decreasing (a = 1)'] >= 0 else 'Negative',
        'Positive' if welfare.iloc[0]['Decreasing (a = 2)'] >= 0 else 'Negative',
        'Positive' if welfare.iloc[0]['Car dependant (Q4)'] >= 0 else 'Negative',
        'Positive' if welfare.iloc[0]['Rawlsian & car dependant (Q4)'] >= 0 else 'Negative',
        'Positive' if welfare.iloc[0]['Vertical & Horizontal'] >= 0 else 'Negative',
    ]
})

latex_lines = [
    r'\begin{tabular}{p{4cm} p{8cm} p{2.5cm}}',
    r'\hline',
    r'Social welfare function & Welfare weights & Result \\',
    r'\hline',
]

for _, row in welfare_latex_table.iterrows():
    latex_lines.append(
        "{} & {} & {} {}".format(
           row['Social welfare function'],
            row['Welfare weights'],
            row['Result'],
            r'\\',
        )
    )

latex_lines.extend([
    r'\hline',
    r'\end{tabular}',
])

welfare_latex = '\n'.join(latex_lines)
print(welfare_latex)
welfare_latex_table